In [2]:
# Core Libraries
import pandas as pd
import numpy as np

# Sklearn Tools
from sklearn.model_selection import train_test_split

# Transformers and Datasets
from transformers import (AutoTokenizer,
                          AutoModelForSequenceClassification,
                          TrainingArguments, Trainer
)
from datasets import Dataset
import evaluate

# PyTorch
import torch

d:\sentiment and sarcasm analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv('./data/IMDB Dataset.csv')

In [4]:
# Show the first 5 rows
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [5]:
# Keep only Positive and Negative samples
df['sentiment'] = df['sentiment'].map({'negative': 0, 'positive': 1})

In [6]:
# Step 1: Split into train (80%) and test (20%)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['sentiment'])

# Step 2: Split train into train (80% of 80%) and val (20% of 80%) → 64% train, 16% val
train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, stratify=train_df['sentiment'])

# Show sizes
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

Train size: 32000
Validation size: 8000
Test size: 10000


In [7]:
# 1. Load tokenizer and model
model_name = "microsoft/deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 102/102 [00:00<00:00, 41367.14it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.weight                       | MISSING    | 
classifier

In [8]:
# 2. Convert pandas DataFrames to HuggingFace Datasets
train_dataset = Dataset.from_pandas(train_df[['review', 'sentiment']], preserve_index=False)
val_dataset = Dataset.from_pandas(val_df[['review', 'sentiment']], preserve_index=False)

In [9]:
# 3. Tokenization
def tokenize_function(example):
    return tokenizer(example['review'], truncation=True, padding='max_length', max_length=512)

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Remove original text column
train_dataset = train_dataset.remove_columns(['review'])
val_dataset = val_dataset.remove_columns(['review'])

# Rename label column
train_dataset = train_dataset.rename_column("sentiment", "labels")
val_dataset = val_dataset.rename_column("sentiment", "labels")

# Set format for PyTorch
train_dataset.set_format("torch")
val_dataset.set_format("torch")

Map: 100%|██████████| 8000/8000 [00:01<00:00, 4440.90 examples/s]


In [10]:
# 4. Define evaluation metric
accuracy = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return accuracy.compute(predictions=predictions, references=labels)

In [12]:
from transformers import TrainingArguments, Trainer

# 5. TrainingArguments
training_args = TrainingArguments(
    output_dir="./results_deberta",
    eval_strategy="epoch",
    save_strategy="epoch",
    report_to="none",
    learning_rate=2e-5,
    per_device_train_batch_size=8, # Nếu bị lỗi "Out of Memory", hãy giảm xuống 8
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,               # Hiển thị log thường xuyên hơn để theo dõi
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
     # THAY ĐỔI Ở ĐÂY:
    fp16=False,       # Tắt FP16 để hết lỗi ValueError
    bf16=True,        # Bật BF16 (Cực tốt cho RTX 4050)
    tf32=True,        # Bật TF32 (Tăng tốc độ tính toán trên nhân Tensor)
    dataloader_num_workers=0        # Giữ bằng 0 để tránh lỗi đa tiến trình trên Windows
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [13]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    data_collator=data_collator, # Sử dụng cái này thay cho tokenizer trực tiếp
)

In [21]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.000000,nan,0.500000
2,0.000000,nan,0.500000
3,0.000000,nan,0.500000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.33s/it]
There were missing keys in the checkpoint model loaded: ['deberta.embeddings.LayerNorm.weight', 'deberta.embeddings.LayerNorm.bias', 'deberta.encoder.layer.0.attention.output.LayerNorm.weight', 'deberta.encoder.layer.0.attention.output.LayerNorm.bias', 'deberta.encoder.layer.0.output.LayerNorm.weight', 'deberta.encoder.layer.0.output.LayerNorm.bias', 'deberta.encoder.layer.1.attention.output.LayerNorm.weight', 'deberta.encoder.layer.1.attention.output.LayerNorm.bias', 'deberta.encoder.layer.1.output.LayerNorm.weight', 'deberta.encoder.layer.1.output.LayerNorm.bias', 'deberta.encoder.layer.2.attention.output.LayerNorm.weight', 'deberta.encoder.layer.2.attention.output.LayerNorm.bias', 'deberta.encoder.layer.2.output.LayerNorm.weight', 'deberta.encoder.layer.2.output.LayerNorm.bias', 'deberta.encoder.layer.3.attention.output.LayerNorm.weight', 'deberta.encoder.layer.3.attention.output.LayerNorm.bias', 'deberta.encoder.la

TrainOutput(global_step=12000, training_loss=0.0034384642442067464, metrics={'train_runtime': 3578.5101, 'train_samples_per_second': 26.827, 'train_steps_per_second': 3.353, 'total_flos': 1.2717323255808e+16, 'train_loss': 0.0034384642442067464, 'epoch': 3.0})

In [14]:
# Convert test_df to HuggingFace Dataset
test_dataset = Dataset.from_pandas(test_df[['review', 'sentiment']])

# Apply tokenization
test_dataset = test_dataset.map(tokenize_function, batched=True)

# Remove the raw text column
test_dataset = test_dataset.remove_columns(['review'])

# Rename the target column to 'labels'
test_dataset = test_dataset.rename_column("sentiment", "labels")

# Set format to PyTorch tensors
test_dataset.set_format("torch")

# Make predictions using the trained model
predictions_output = trainer.predict(test_dataset)

# Extract true labels and predicted labels
y_pred = np.argmax(predictions_output.predictions, axis=1)
y_true = predictions_output.label_ids

Map: 100%|██████████| 10000/10000 [00:02<00:00, 4236.94 examples/s]


In [15]:
# Convert logits to probabilities
probs = torch.nn.functional.softmax(torch.tensor(predictions_output.predictions), dim=1).numpy()

# Create DataFrame with the results
results_df = pd.DataFrame({
    'review': test_df['review'],
    'true_label': y_true,
    'predicted_label': y_pred,
    'prob_negative': probs[:, 0],
    'prob_positive': probs[:, 1]
})

# Save to CSV
results_df.to_csv("deberta_preds.csv", index=False)

print("✅ Predictions and probabilities saved to deberta_preds.csv")

✅ Predictions and probabilities saved to deberta_preds.csv
